In [1]:
import sys
sys.path.insert(1, '/Users/amjonz/Documents/GitHub/mesher/src')
import mesher as msh
import gmsh
import numpy as np
import pyvista as pv

In [7]:

surface = pv.Plane(center=(0.5,0.5,0.5), direction=(0,1,0), i_size=1., j_size=1., i_resolution=10, j_resolution=10)

points = surface.points
faces = surface.faces.reshape((-1, 4))[:, 1:]
points.round(2), faces 

#pv.Plane?

(array([[1. , 0.5, 1. ],
        [1. , 0.5, 0.9],
        [1. , 0.5, 0.8],
        [1. , 0.5, 0.7],
        [1. , 0.5, 0.6],
        [1. , 0.5, 0.5],
        [1. , 0.5, 0.4],
        [1. , 0.5, 0.3],
        [1. , 0.5, 0.2],
        [1. , 0.5, 0.1],
        [1. , 0.5, 0. ],
        [0.9, 0.5, 1. ],
        [0.9, 0.5, 0.9],
        [0.9, 0.5, 0.8],
        [0.9, 0.5, 0.7],
        [0.9, 0.5, 0.6],
        [0.9, 0.5, 0.5],
        [0.9, 0.5, 0.4],
        [0.9, 0.5, 0.3],
        [0.9, 0.5, 0.2],
        [0.9, 0.5, 0.1],
        [0.9, 0.5, 0. ],
        [0.8, 0.5, 1. ],
        [0.8, 0.5, 0.9],
        [0.8, 0.5, 0.8],
        [0.8, 0.5, 0.7],
        [0.8, 0.5, 0.6],
        [0.8, 0.5, 0.5],
        [0.8, 0.5, 0.4],
        [0.8, 0.5, 0.3],
        [0.8, 0.5, 0.2],
        [0.8, 0.5, 0.1],
        [0.8, 0.5, 0. ],
        [0.7, 0.5, 1. ],
        [0.7, 0.5, 0.9],
        [0.7, 0.5, 0.8],
        [0.7, 0.5, 0.7],
        [0.7, 0.5, 0.6],
        [0.7, 0.5, 0.5],
        [0.7, 0.5, 0.4],


In [3]:
tracktest = DimensionIDTracker()
tracktest.add_entry([(2,1)], ["this one!"])

tracktest.lookup_name((2,1))

tracktest.update_entries([(2,1)], [[(2,3),(2,4)]])

NameError: name 'DimensionIDTracker' is not defined

In [58]:
######### THIS IS WORKING WITH THE EXCEPTION OF THE OUTER BOXES ########### replicating to get an idea about how to continue 




box_dimensions = (0,1,0,1,0,1)

gmsh.initialize()
gmsh.model.add("divided_volume")

    # Define a rectangular box in Gmsh using OCC kernel
    #length, width, height = box_dimensions
    #box = gmsh.model.occ.add_box(0, 0, 0, length, width, height)
xmin,xmax,ymin,ymax,zmin,zmax = box_dimensions
box = gmsh.model.occ.add_box(xmin, ymin, zmin, xmax, ymax, zmax)
    # Synchronize after adding the box
face_names =  ['West', 'East', 'South', 'North', 'Base', 'Top']
tracker = DimensionIDTracker()



box_2dim = gmsh.model.occ.get_entities(dim=2)

tracker.add_entry(box_2dim, face_names)


plane_surface = pv.Plane(center=(0.5,0.5,0.5), direction=(0,1,0), i_size=1.0, j_size=1.0, i_resolution=2, j_resolution=2)
surface = msh.clip_polydata(plane_surface, xmin, xmax, ymin, ymax, zmin, zmax)
points = surface.points
faces = surface.faces.reshape((-1, 4))[:, 1:]
#points.round(2), faces 

gmsh_points = []
#points = [(0,0,.5), (0,1,.5), (1,1,.5), (1,0,.5)]
for pt in points:
    gmsh_points.append(gmsh.model.occ.add_point(pt[0], pt[1], pt[2]))
print("Points ",gmsh_points)

gmsh_curves = []
for face in faces:
        lines = []
        for j in range(len(face)):
            p1 = gmsh_points[face[j]]
            p2 = gmsh_points[face[(j + 1) % len(face)]]
            lines.append(gmsh.model.occ.add_line(p1, p2))
            print(f"line {lines} {j}")
        loop = gmsh.model.occ.add_curve_loop(lines)
        print("loop ", loop)
        surface_tag = gmsh.model.occ.add_plane_surface([loop])
        print("adding surface tag : ", surface_tag)
        gmsh_curves.append(surface_tag)

print("Curves :", gmsh_curves)
plane1 = 'Plane_1'
namer = [plane1] * len(gmsh_curves)
print(namer)
tracker.add_entry([(2, plane_parts) for plane_parts in gmsh_curves], namer)
"""
gmsh_curves = []
        for face in faces:
            lines = []
            for j in range(len(face)):
                p1 = gmsh_points[face[j]]
                p2 = gmsh_points[face[(j + 1) % len(face)]]
                lines.append(gmsh.model.occ.add_line(p1, p2))
            loop = gmsh.model.occ.add_curve_loop(lines)
            surface_tag = gmsh.model.occ.add_plane_surface([loop])
            gmsh_curves.append(surface_tag)
"""







gmsh.model.occ.synchronize()

surf_dict = {0:'West', 1:'East', 2:'South', 3:'North', 4:'Base', 5:'Top'}
all_2dim = gmsh.model.occ.get_entities(dim=2) #box creation order (ZY- , ZY+ , ZX- , ZX+ , XY- , XY+) or (W, E, S, N, Dn, Up)
print("Box 2 dim tags tuples: ", all_2dim)
box_3dim = gmsh.model.occ.get_entities(dim=3) #original box
gmsh.model.occ.synchronize()
#box2_dimtags = filter_tuples_by_first_entry(box_2dim, 2)   # this function only returns the 2D surface tags


box_2dim = gmsh.model.occ.get_entities(dim=2)
boxtags = [tags[1] for tags in box_2dim]

surf_dict = "surf"

print("Box tags without dimension: ",boxtags)

all_surf = gmsh.model.occ.get_entities(dim=2)
print(all_surf)
vol = [(3, box)]  # The box is initially the entire volume
tracker.add_entry(vol, ["Whole_Cube"])
print("gmsh _curves ",gmsh_curves)
#partitions = all_surf.extend(vol[0])
#print(partitions)
#tracker.add_entries()
box_surfaces = all_surf+vol

ovv, ov = gmsh.model.occ.fragment(box_surfaces, all_surf, removeTool=True, removeObject=True) #[(2, plane_parts) for plane_parts in gmsh_curves]
gmsh.model.occ.synchronize()
for e in zip(box_surfaces, ov):
    print("parent " + str(e[0]) + " -> child " + str(e[1]))
print("ov = ",len(ov),ov)
print("ovv = ",len(ovv),ovv)

print("Box Surfaces = ", len(box_surfaces), box_surfaces)
tracker.update_entries(box_surfaces+all_surf, ov)

counts = tracker.get_name_counts(by_dimension=True)
#counts[2]

tags_3dim = gmsh.model.occ.get_entities(dim=3)
tags_2dim = gmsh.model.occ.get_entities(dim=2)
fragtags_dim2 = msh.filter_tuples_by_first_entry(ovv, 2)


print('Dim3 Tags: ', tags_3dim)
for tag in tags_3dim:
     print('current tag 3', tag)
     gmsh.model.add_physical_group(3, [tag[1]], tag=tag[1])
     gmsh.model.set_physical_name(3, tag[1], f"{tracker.lookup_name(tag)}_{tag[1]:04d}")
     counts[3][tracker.lookup_name(tag)] = counts[3][tracker.lookup_name(tag)]-1
print("Dim 2 frag tags: ",tags_2dim)
for tag in tags_2dim:
     print('current tag 2', tag)
     gmsh.model.add_physical_group(2, [tag[1]], tag=tag[1])
     gmsh.model.set_physical_name(2, tag[1], f"{tracker.lookup_name(tag)}_{counts[2][tracker.lookup_name(tag)]+10000}")
     counts[2][tracker.lookup_name(tag)] = counts[2][tracker.lookup_name(tag)]-1

#counts[3].keys-1
print(counts)

gmsh.model.occ.synchronize


gmsh.model.mesh.generate(3)
gmsh.write('../test_results/unit_box.msh')
gmsh.finalize()



######## THIS IS REPLICATED BELOW TO TRY SOMETHING ELSE ####### 


Points  [9, 10, 11, 12, 13, 14, 15, 16, 17]
line [13] 0
line [13, 14] 1
line [13, 14, 15] 2
loop  7
adding surface tag :  7
line [16] 0
line [16, 17] 1
line [16, 17, 18] 2
loop  8
adding surface tag :  8
line [19] 0
line [19, 20] 1
line [19, 20, 21] 2
loop  9
adding surface tag :  9
line [22] 0
line [22, 23] 1
line [22, 23, 24] 2
loop  10
adding surface tag :  10
line [25] 0
line [25, 26] 1
line [25, 26, 27] 2
loop  11
adding surface tag :  11
line [28] 0
line [28, 29] 1
line [28, 29, 30] 2
loop  12
adding surface tag :  12
line [31] 0
line [31, 32] 1
line [31, 32, 33] 2
loop  13
adding surface tag :  13
line [34] 0
line [34, 35] 1
line [34, 35, 36] 2
loop  14
adding surface tag :  14
Curves : [7, 8, 9, 10, 11, 12, 13, 14]
['Plane_1', 'Plane_1', 'Plane_1', 'Plane_1', 'Plane_1', 'Plane_1', 'Plane_1', 'Plane_1']
Box 2 dim tags tuples:  [(2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (2, 11), (2, 12), (2, 13), (2, 14)]
Box tags without dimension:  [1, 2, 

In [52]:
tracker.lookup_name((2,22))
#tracker.get_hierarchy()
#tracker.get_name_counts(by_dimension=True)

In [ ]:
def export_divided_gmsh_volume_Compound_surface_accounting(
    surfaces, 
    box_dimensions, 
    output_filename="output.msh",
    surface_names={},
    field_size=500
    
):
    """
    Divides a rectangular box using a series of PyVista surfaces and exports a Gmsh volume mesh.
    Groups and labels surfaces created after each division.

    Parameters:
    surfaces (list): List of PyVista PolyData surfaces to divide the box.
    box_dimensions (tuple): The dimensions of the rectangular box (length, width, height).
    output_filename (str): The filename for the output .msh file.
    """

    gmsh.initialize()
    gmsh.model.add("divided_volume")

    # Define a rectangular box in Gmsh using OCC kernel
    #length, width, height = box_dimensions
    #box = gmsh.model.occ.add_box(0, 0, 0, length, width, height)
    xmin,xmax,ymin,ymax,zmin,zmax = box_dimensions
    box = gmsh.model.occ.add_box(xmin, ymin, zmin, xmax, ymax, zmax)
    # Synchronize after adding the box
    gmsh.model.occ.synchronize()
    
    #gmsh.model.mesh.setSize(gmsh.model.getBoundary([(3, box)]), 200)

    # Alternatively, you can set a field-based size using a background field
    # Field 1: Uniform mesh size over the whole domain
    #gmsh.model.mesh.field.add("Constant", 1)
    #gmsh.model.mesh.field.setNumber(1, "VIn", 200)  # uniform element size
    #gmsh.model.mesh.field.setAsBackgroundMesh(1)
  
    
    # Convert PyVista surfaces to Gmsh geometries
    gmsh_surfaces = []
    for i, surface in enumerate(surfaces):
        points = surface.points
        faces = surface.faces.reshape((-1, 4))[:, 1:]  # Assuming triangular faces

        # Add points to Gmsh
        gmsh_points = []
        for pt in points:
            gmsh_points.append(gmsh.model.occ.add_point(pt[0], pt[1], pt[2]))

        # Add faces as Gmsh surfaces
        gmsh_curves = []
        for face in faces:
            lines = []
            for j in range(len(face)):
                p1 = gmsh_points[face[j]]
                p2 = gmsh_points[face[(j + 1) % len(face)]]
                lines.append(gmsh.model.occ.add_line(p1, p2))
            loop = gmsh.model.occ.add_curve_loop(lines)
            surface_tag = gmsh.model.occ.add_plane_surface([loop])
            gmsh_curves.append(surface_tag)

        # Store the surface tags for later use
        gmsh_surfaces.append(gmsh_curves)
    fragment_ov = []
    fragment_ovv = []
    # Synchronize OCC geometry definitions
    gmsh.model.occ.synchronize()
    all_fragmenting_surfaces = []
    all_fragmenting_surface_names = []
    surface_physical_names = []
    compound_surface_id_idx = []
    # Use the surfaces to divide the rectangular box into different zones
    partitions = [(3, box)]  # The box is initially the entire volume
    for surf_idx in range(len(surface_names)):
        print("Surf idx: " + str(surf_idx))
        SurfName = surface_names[surf_idx]
        # Fragment the current partitions using the new surface
        ovv, ov = gmsh.model.occ.fragment(partitions, [(2, surface_tag) for surface_tag in gmsh_surfaces[surf_idx]], removeTool=True, removeObject=True)
        gmsh.model.occ.synchronize()

        print(SurfName)
        dim2_surface_tags = filter_tuples_by_first_entry(ovv, 2)   # this function only returns the 2D surface tags
        frag_surf_tags = [tags[1] for tags in dim2_surface_tags]
        # Assign physical groups to the new surfaces created by this fragmentation
        all_fragmenting_surface_names = []  # holder of the Physical group names to output later
        for idx, surf in enumerate(frag_surf_tags, start=1):
            
            group_tag = 1000 * (surf_idx + 1) + int(surf)  # Create a unique identifier for the surface group (six digit is surfaces and four digit is partition)
            gmsh.model.add_physical_group(2, [surf], tag=group_tag)
            gmsh.model.set_physical_name(2, group_tag, f"{SurfName}_{idx:04d}")
            all_fragmenting_surface_names.append(f"{SurfName}_{idx:04d}") # append the name to the name list, will be added later to the surface_physical_names list
            if idx == 1:
                compound_surface_id_idx.append(gmsh.model.getEntitiesForPhysicalName(f"{SurfName}_{idx:04d}")) # this is a strage OCC behavior, the compound surface is indexed 100000 plus this first new entity, not exactly sure why
            # Get updated partitions
            partitions = gmsh.model.occ.get_entities(dim=3)

            gmsh.model.occ.synchronize()
        surface_physical_names.append(all_fragmenting_surface_names) # These lists are coded in order by the surface names list input (the surfaces are in the names too so it should be obvious) 
        
        
        # create a compound surface from the fragmented surface tags, use the <**surface group**000> tag (e.g. 1000, 2000, etc) for the entity id (compounding surface entities start numbering at 1)     
        #gmsh.model.mesh.setCompound(2, frag_surf_tags)
        #surf_group_tag = 1000 * (surf_idx + 1)
        #gmsh.model.add_physical_group(2, [compound_surf], tag=surf_group_tag)
        #gmsh.model.set_physical_name(2, surf_group_tag, f"{SurfName}")



        #gmsh.model.mesh.setCompound(2, frag_surf_tags)

        ''' Moving this down to after the generate command
        print(compound_surface_id_idx)
        compound_tag = 100000 + compound_surface_id_idx[0][1]
        print("compound surf tag= " + str(compound_tag))
        new_tag = (100000 * (surf_idx+1))
        gmsh.model.add_physical_group(2, [compound_tag], tag=new_tag)
        gmsh.model.set_physical_name(2, new_tag, SurfName)
        #gmsh.model.setEntityName(2, compound_tag, name=f"{SurfName}")
        '''


        fragment_ov.append(ov)
        fragment_ovv.append(ovv)
        all_fragmenting_surfaces.extend(ovv)
    # Collect surfaces that are not on fragmenting the volume, these should be exterior 
    #all_surfaces = [tags[1] for tags in gmsh.model.get_entities(2)]
    gmsh.model.mesh.reclassifyNodes()
    all_surfaces = gmsh.model.get_entities(2)   # Experimental
    exterior_surfaces = unique_points(all_surfaces, all_fragmenting_surfaces)

    volume_names = []
    exterior_surface_names = []
    for idx, (dim, surf) in enumerate(exterior_surfaces, start=1):
        print("ext surfaces " + str(surf))
        group_tag = 10000 + int(surf)  # Create a unique identifier for the surface group (six digit is surfaces and four digit is partition)
        gmsh.model.add_physical_group(2, [surf], tag=group_tag)
        gmsh.model.set_physical_name(2, group_tag, f"Ext_Surf_{idx:02d}")
        exterior_surface_names.append(f"Ext_Surf_{idx:02d}")

    # Assign unique identifiers to each zone (volume)
    for idx, (dim, volume) in enumerate(partitions, start=1):
        gmsh.model.add_physical_group(3, [volume], tag=idx)
        gmsh.model.set_physical_name(3, idx, f"Volume_{idx:02d}")
        volume_names.append(f"Volume_{idx:02d}")
    gmsh.model.occ.synchronize()
   
   
   
    # Generate the mesh
    pnt_entities = gmsh.model.occ.get_entities(dim=0)

    gmsh.model.mesh.setSize(pnt_entities, field_size) #field size determines the maximum distance between point node elements

    # Alternatively, you can set a field-based size using a background field
    # Field 1: Uniform mesh size over the whole domain
    #gmsh.model.mesh.field.add("Constant", 1)
    #gmsh.model.mesh.field.setNumber(1, "VIn", 200)  # uniform element size
    #gmsh.model.mesh.field.setAsBackgroundMesh(1)

    gmsh.model.mesh.generate(3)   # generate 3D mesh
    #for surf_idx in range(len(compound_surface_id_idx)):
    #    print(compound_surface_id_idx)
    #    compound_tag = 100000 + compound_surface_id_idx[surf_idx][0][1]
    #    print("compound surf tag= " + str(compound_tag))
    #    new_tag = (100000 * (surf_idx+1))
    #    gmsh.model.add_physical_group(2, [compound_tag], tag=new_tag)
    #    gmsh.model.set_physical_name(2, new_tag, surface_names[surf_idx])
        #gmsh.model.setEntityName(2, compound_tag, name=f"{SurfN
    # Export mesh to file
    gmsh.write(output_filename)
    gmsh.finalize()
    print(f"Mesh exported to {output_filename}")
    return fragment_ov, fragment_ovv, gmsh_surfaces, surface_physical_names, volume_names, exterior_surface_names

In [18]:
from collections import defaultdict

class DimensionIDTracker:
    def __init__(self):
        self.data = {}  # Stores {(dim, id): name}
        self.parent_child_map = defaultdict(list)  # Stores parent-child relationships
    
    def add_entry(self, dim_ids, names):
        """
        Adds multiple entries to the tracker.
        :param dim_ids: List of tuples [(dimension, id), ...]
        :param names: List of strings [name, ...]
        """
        if len(dim_ids) != len(names):
            raise ValueError("Dimension ID list and names list must be of the same length.")
        
        for dim_id, name in zip(dim_ids, names):
            if dim_id[0] not in {1, 2, 3}:
                raise ValueError("Dimension must be 1, 2, or 3")
            self.data[dim_id] = name
    
    def update_entries(self, parent_entries, child_entries):
        """
        Updates the tracker based on transformations.
        :param orig_entries: List of tuples [(orig_dim, orig_id), ...]
        :param new_entries: List of tuples [(new_dim, new_id), ...]
        """
        if len(parent_entries) != len(child_entries): # this is not correct as we may have more than one resulting surface from a fragment
            raise ValueError("Original entries and new entries lists must be of the same length.")
        
        
        for orig, children in zip(parent_entries, child_entries):
            print("Orig: ",orig," Children: ",children)
            if orig not in self.data:
                    raise KeyError(f"Original entry {orig} not found.")
            parent_name = self.data[orig]
            print("Children: ",children)
            if children == []:
                remove_name = "remove"
                self.data[orig] = remove_name   
            else:
                for new in children:
                    print(f"Adding {new} : {parent_name}")
                    #parent_name = self.data[orig]
                    self.data[new] = parent_name  # Inherit the name
                    self.parent_child_map[orig].append(new)  # Track the split
    
    def get_hierarchy(self):
        """
        Returns the parent-child hierarchy as a dictionary.
        """
        return dict(self.parent_child_map)
    
    def lookup_name(self, dim_id):
        """
        Looks up the name associated with a given dimension ID.
        :param dim_id: Tuple (dimension, id)
        :return: Name string if found, otherwise None
        """
        return self.data.get(dim_id, None)
    
    def get_name_counts(self, by_dimension=False):
        """
        Returns a count of each unique name in the dictionary.
        If by_dimension is True, returns a nested dictionary with counts per dimension.
        """
        if by_dimension:
            name_counts = defaultdict(lambda: defaultdict(int))
            for (dim, _), name in self.data.items():
                name_counts[dim][name] += 1
            return {dim: dict(names) for dim, names in name_counts.items()}
        else:
            name_counts = defaultdict(int)
            for name in self.data.values():
                name_counts[name] += 1
            return dict(name_counts)
    
    
    
    def __repr__(self):
        return f"Data: {self.data}\nParent-Child Map: {dict(self.parent_child_map)}"


In [53]:
gmsh.model.occ.fragment??

Signature:
gmsh.model.occ.fragment(
    objectDimTags,
    toolDimTags,
    tag=-1,
    removeObject=True,
    removeTool=True,
)
Source:   
        @staticmethod
        def fragment(objectDimTags, toolDimTags, tag=-1, removeObject=True, removeTool=True):
            """
            gmsh.model.occ.fragment(objectDimTags, toolDimTags, tag=-1, removeObject=True, removeTool=True)

            Compute the boolean fragments (general fuse) resulting from the
            intersection of the entities `objectDimTags' and `toolDimTags' (given as
            vectors of (dim, tag) pairs) in the OpenCASCADE CAD representation, making
            all interfaces conformal. When applied to entities of different dimensions,
            the lower dimensional entities will be automatically embedded in the higher
            dimensional entities if they are not on their boundary. Return the
            resulting entities in `outDimTags'. If `tag' is positive, try to set the
            tag explicitly (on